# LightGBM Model for Credit Risk Prediction

## 1. Setup

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from seqcredit_model.credit_model import (
    CreditRiskDataLoader, LightGBMModel, ModelEvaluator, set_random_seeds
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

set_random_seeds(42)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print('Setup complete.')

## 2. Data Preparation

In [ ]:
loader = CreditRiskDataLoader(
    features_path=os.path.join(project_root, 'data/user_features.csv'),
    summaries_path=os.path.join(project_root, 'data/user_labels.csv'),
    transactions_dir=os.path.join(project_root, 'data/user_transactions'),
)

static_data = loader.prepare_static_splits()

print(f"Training samples: {len(static_data['y_train'])}")
print(f"Test samples: {len(static_data['y_test'])}")
print(f"Default rate (train): {static_data['y_train'].mean():.4f}")
print(f"Default rate (test): {static_data['y_test'].mean():.4f}")
print(f"Features: {len(static_data['feature_names'])}")

## 3. Model Training

In [ ]:
from sklearn.model_selection import train_test_split as tts

X_tr, X_val, y_tr, y_val = tts(
    static_data['X_train_scaled'], static_data['y_train'],
    test_size=0.15, stratify=static_data['y_train'], random_state=42
)

lgbm_model = LightGBMModel(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    num_leaves=31,
    class_weight='balanced'
)
lgbm_model.fit(X_tr, y_tr, X_val=X_val, y_val=y_val)

print('\nLightGBM - 5-Fold Cross-Validation:')
lgbm_cv = lgbm_model.cross_validate(static_data['X_train_scaled'], static_data['y_train'])
for metric, values in lgbm_cv.items():
    print(f'  {metric}: {np.mean(values):.4f} +/- {np.std(values):.4f}')

## 4. Evaluation

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

lgbm_proba = lgbm_model.predict_proba(static_data['X_test_scaled'])

print('LightGBM - Test Set Performance:')
print(f'  AUC-ROC: {roc_auc_score(static_data["y_test"], lgbm_proba):.4f}')
print(f'  AUC-PR: {average_precision_score(static_data["y_test"], lgbm_proba):.4f}')

print('\nClassification Report:')
print(classification_report(static_data['y_test'], (lgbm_proba >= 0.5).astype(int), target_names=['Non-Default', 'Default']))

## 5. Feature Importance

In [ ]:
lgbm_importance = lgbm_model.get_feature_importance(static_data['feature_names'])

fig, ax = plt.subplots(figsize=(10, 8))
top_imp = lgbm_importance.head(15)
ax.barh(range(len(top_imp)), top_imp['importance'], color='#f39c12')
ax.set_yticks(range(len(top_imp)))
ax.set_yticklabels(top_imp['feature'], fontsize=9)
ax.set_xlabel('Importance')
ax.set_title('LightGBM - Top 15 Feature Importances')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Model Comparison

In [ ]:
evaluator = ModelEvaluator(static_data['y_test'])
evaluator.add_model('LightGBM', lgbm_proba)

comparison = evaluator.get_comparison_table()
print('\nModel Comparison (Test Set):')
print(comparison.round(4).to_string())

In [ ]:
evaluator.plot_roc_curves()
plt.show()

In [ ]:
evaluator.plot_pr_curves()
plt.show()

In [ ]:
evaluator.plot_confusion_matrices()
plt.show()